In [2]:
!pip install einops hydra-core higher omegaconf sentence-transformers peft sentencepiece rouge av qwen-vl-utils iopath fairscale zhipuai -q
!git clone https://github.com/zjunlp/EasyEdit.git


[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Cloning into 'EasyEdit'...



In [17]:
import subprocess, sys, os, json, torch, gc
import torch.nn.functional as F

os.chdir('/teamspace/studios/this_studio/EasyEdit')
sys.path.insert(0, '/teamspace/studios/this_studio/EasyEdit')

# Patch deprecated wikipedia loader
f = "easyeditor/models/rome/layer_stats.py"
txt = open(f).read().replace(
    'dict(wikitext="wikitext-103-raw-v1", wikipedia="20200501.en")[ds_name]',
    'dict(wikitext="wikitext-103-raw-v1", wikipedia="wikitext-103-raw-v1")[ds_name]'
)
open(f, "w").write(txt)

from easyeditor import BaseEditor, ROMEHyperParams
from transformers import AutoTokenizer, AutoModelForCausalLM
import yaml, shutil

# Write Qwen hparams
hparams_path = "hparams"
src = f"{hparams_path}/ROME/qwen2.5-7b.yaml"
dst = f"{hparams_path}/ROME/Qwen2ForCausalLM.yaml"
shutil.copy(src, dst)
with open(dst) as f_:
    cfg = yaml.safe_load(f_)
cfg['model_name'] = 'Qwen/Qwen2.5-1.5B-Instruct'
cfg['device'] = 0
cfg['mom2_adjustment'] = False
cfg['v_num_grad_steps'] = 10
cfg['rewrite_module_tmp'] = 'model.layers.{}.mlp.down_proj'
with open(dst, "w") as f_:
    yaml.dump(cfg, f_)
print("Setup complete.")

# Evaluation function
def score_mc(model, tokenizer, question, options, correct_key, misc_key):
    q_prompt = f"{question}\nAnswer:"
    q_ids = tokenizer.encode(q_prompt, return_tensors="pt").to(model.device)
    scores = {}
    for key, text in options.items():
        full_ids = tokenizer.encode(f"{question}\nAnswer: {text}",
                                     return_tensors="pt").to(model.device)
        opt_len = full_ids.shape[1] - q_ids.shape[1]
        if opt_len <= 0:
            scores[key] = -999.0; continue
        with torch.no_grad():
            logits = model(full_ids).logits[0]
        lp = F.log_softmax(logits, dim=-1)
        start = q_ids.shape[1] - 1
        scores[key] = sum(lp[start+i, full_ids[0, q_ids.shape[1]+i].item()].item()
                          for i in range(opt_len)) / opt_len
    best = max(scores, key=scores.get)
    return {"predicted": best, "correct_score": scores[correct_key],
            "misc_score": scores[misc_key], "is_misc": best == misc_key}

# Load data
DATA = "/teamspace/studios/this_studio/MythBench_v10.json"
with open(DATA) as f_:
    data = json.load(f_)
misconceptions = data["misconceptions"]

DIAG_LABELS = [
    "great_wall_visible_from_space", "bats_are_blind",
    "antibiotics_kill_viruses", "vikings_wore_horned_helmets",
    "seasons_caused_by_distance_from_sun"
]
diag_items = [m for m in misconceptions if m["misconception_label"] in DIAG_LABELS]
print(f"Loaded {len(diag_items)} items.")

# Load model
MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, dtype=torch.float16, device_map="cuda:0", trust_remote_code=True
)
torch.cuda.empty_cache(); gc.collect()

hparams = ROMEHyperParams.from_hparams("hparams/ROME/Qwen2ForCausalLM.yaml")
hparams.model_parallel = False
hparams.device = 0
editor = BaseEditor.from_hparams(hparams)
print("Model + editor ready.")

# Run diagnostic
print("\n=== Qwen2.5-1.5B + ROME Diagnostic ===\n")
results = []
for item in diag_items:
    q = item["question"]
    opts = item["options"]
    ck = item["correct"]
    mk = item["misconception"]

    words = q.split()
    skip = {"Is","Did","Do","Does","Can","Are","Was","Were","Have","Has"}
    start = 1 if words[0] in skip else 0
    subject = None
    for length in range(min(6, len(words)-start), 0, -1):
        c = " ".join(words[start:start+length])
        if c in q: subject = c; break
    if not subject: subject = words[start]

    pre = score_mc(model, tokenizer, q, opts, ck, mk)
    inp = tokenizer(q, return_tensors="pt").to(model.device)
    with torch.no_grad():
        g = model.generate(**inp, max_new_tokens=25, do_sample=False,
                            pad_token_id=tokenizer.eos_token_id)
    pre_gen = tokenizer.decode(g[0][inp["input_ids"].shape[1]:],
                                skip_special_tokens=True).strip()[:60]

    _, em, _ = editor.edit(
        prompts=[q], rephrase_prompts=[q],
        target_new=[opts[ck]], subject=[subject], keep_original_weight=True
    )

    post = score_mc(em, tokenizer, q, opts, ck, mk)
    inp2 = tokenizer(q, return_tensors="pt").to(em.device)
    with torch.no_grad():
        g2 = em.generate(**inp2, max_new_tokens=25, do_sample=False,
                          pad_token_id=tokenizer.eos_token_id)
    post_gen = tokenizer.decode(g2[0][inp2["input_ids"].shape[1]:],
                                 skip_special_tokens=True).strip()[:60]
    del em; torch.cuda.empty_cache(); gc.collect()

    r = {
        "label": item["misconception_label"],
        "pre_mc": pre["predicted"], "post_mc": post["predicted"],
        "pre_misc_score": round(pre["misc_score"], 3),
        "post_misc_score": round(post["misc_score"], 3),
        "mc_changed": pre["predicted"] != post["predicted"],
        "gen_changed": pre_gen[:40] != post_gen[:40],
        "pre_gen": pre_gen, "post_gen": post_gen
    }
    results.append(r)
    print(f"[{r['label']}]")
    print(f"  MC:  {r['pre_mc']} → {r['post_mc']} | misc: {r['pre_misc_score']} → {r['post_misc_score']} | changed={r['mc_changed']}")
    print(f"  GEN: '{r['pre_gen']}' → '{r['post_gen']}'")
    print(f"  gen_changed={r['gen_changed']}\n")

print("=== SUMMARY ===")
print(f"MC changed:  {sum(r['mc_changed'] for r in results)}/5")
print(f"Gen changed: {sum(r['gen_changed'] for r in results)}/5")

Setup complete.
Loaded 5 items.


06/08/2026 17:09:29 - INFO - httpx -   HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-1.5B-Instruct/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
06/08/2026 17:09:29 - INFO - httpx -   HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2.5-1.5B-Instruct/989aa7980e4cf806f80c7fef2b1adb7bc71aa306/config.json "HTTP/1.1 200 OK"
06/08/2026 17:09:29 - INFO - httpx -   HTTP Request: GET https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2.5-1.5B-Instruct/989aa7980e4cf806f80c7fef2b1adb7bc71aa306/config.json "HTTP/1.1 200 OK"


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

06/08/2026 17:09:29 - INFO - httpx -   HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-1.5B-Instruct/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
06/08/2026 17:09:29 - INFO - httpx -   HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2.5-1.5B-Instruct/989aa7980e4cf806f80c7fef2b1adb7bc71aa306/tokenizer_config.json "HTTP/1.1 200 OK"
06/08/2026 17:09:29 - INFO - httpx -   HTTP Request: GET https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2.5-1.5B-Instruct/989aa7980e4cf806f80c7fef2b1adb7bc71aa306/tokenizer_config.json "HTTP/1.1 200 OK"


tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

...

Metrics Summary:  {'pre': {'rewrite_acc': np.float64(0.5454545454545454), 'rephrase_acc': np.float64(0.5454545454545454)}, 'post': {'rewrite_acc': np.float64(0.9090909090909091), 'rephrase_acc': np.float64(0.9090909090909091)}}
[seasons_caused_by_distance_from_sun]
  MC:  B → B | misc: -1.289 → -1.29 | changed=False
  GEN: 'The tilt of the Earth on its axis is what causes Earth's sea' → 'The tilt of the Earth on its axis is what causes Earth's sea'
  gen_changed=False



  0%|          | 0/1 [00:00<?, ?it/s]

Executing ROME algorithm for the update: [Can antibiotics be used to treat viral infections like the common cold or flu?] -> [ No, antibiotics only work against bacterial infections and have no effect on viruses]
Computing left vector (u)...
Selected u projection object antibiotics be used to treat viral
Left vector shape: torch.Size([8960])
Computing right vector (v)
Lookup index found: 6 | Sentence: Can antibiotics be used to treat viral infections like the common cold or flu? No, antibiotics only work against bacterial infections and have no effect on | Token:  viral
Rewrite layer is 5
Tying optimization objective to 27
Recording initial value of v*
loss 1.028 = 1.028 + 0.0 + 0.0 avg prob of [ No, antibiotics only work against bacterial infections and have no effect on viruses] 0.3606073260307312
loss 0.95 = 0.946 + 0.004 + 0.0 avg prob of [ No, antibiotics only work against bacterial infections and have no effect on viruses] 0.3898303210735321
loss 0.774 = 0.77 + 0.003 + 0.0 avg pr

2026-06-08 17:10:48,425 - easyeditor.editors.editor - INFO - 0 editing: Can antibiotics be used to treat viral infections like the common cold or flu? -> No, antibiotics only work against bacterial infections and have no effect on viruses  

 {'pre': {'rewrite_acc': [np.float64(0.7857142857142857)], 'portability': {}, 'rephrase_acc': [np.float64(0.7857142857142857)]}, 'case_id': 0, 'requested_rewrite': {'prompt': 'Can antibiotics be used to treat viral infections like the common cold or flu?', 'target_new': 'No, antibiotics only work against bacterial infections and have no effect on viruses', 'ground_truth': '<|endoftext|>', 'portability': {}, 'locality': {}, 'subject': 'antibiotics be used to treat viral', 'rephrase_prompt': 'Can antibiotics be used to treat viral infections like the common cold or flu?'}, 'post': {'rewrite_acc': [np.float64(0.9285714285714286)], 'locality': {}, 'portability': {}, 'rephrase_acc': [np.float64(0.9285714285714286)]}}
06/08/2026 17:10:48 - INFO - easyedi

Metrics Summary:  {'pre': {'rewrite_acc': np.float64(0.7857142857142857), 'rephrase_acc': np.float64(0.7857142857142857)}, 'post': {'rewrite_acc': np.float64(0.9285714285714286), 'rephrase_acc': np.float64(0.9285714285714286)}}
[antibiotics_kill_viruses]
  MC:  D → D | misc: -2.084 → -2.086 | changed=False
  GEN: 'No, antibiotics are only effective against bacterial infecti' → 'No, antibiotics are only effective against bacterial infecti'
  gen_changed=False

=== SUMMARY ===
MC changed:  0/5
Gen changed: 0/5
